In [1]:
import fitz  # PyMuPDF
import base64
import numpy as np
import tempfile
from pathlib import Path
from typing import List
from PIL import Image, ImageOps, ImageFilter, ImageChops
import pytesseract

# Optional heavy deps guarded
try:
    import cv2  # type: ignore
except Exception:
    cv2 = None

try:
    from skimage.filters import threshold_local  # type: ignore
except Exception:
    threshold_local = None

# Default DPI for PDF to image conversion
DPI = 300

def rotate_if_needed(img: Image.Image) -> Image.Image:
    osd = pytesseract.image_to_osd(img)
    # Parse rotation angle
    width, height = img.size

    if height < width:
        rotation_angle = int([line for line in osd.split('\n') if 'Rotate' in line][0].split(':')[1].strip())
       # print(rotation_angle,'rotation_angle')
        rotated_img = img.rotate(-rotation_angle, expand=True)
        return rotated_img
    return img    

def trim_margins_projection(im: Image.Image, margin_percent: float = 0.02) -> Image.Image:
    """Trim margins using projection method"""
    # Convert to grayscale
    gray = ImageOps.grayscale(im)
    img_array = np.array(gray)
    
    # Invert so dark content becomes bright
    inverted = 255 - img_array
    
    # Sum along axes to find content density
    vertical_projection = np.sum(inverted, axis=0)  # Sum each column
    horizontal_projection = np.sum(inverted, axis=1)  # Sum each row
    
    # Find threshold (margin_percent of max value)
    v_threshold = np.max(vertical_projection) * margin_percent
    h_threshold = np.max(horizontal_projection) * margin_percent
    
    # Find where content starts/ends
    v_content = np.where(vertical_projection > v_threshold)[0]
    h_content = np.where(horizontal_projection > h_threshold)[0]
    
    if len(v_content) > 0 and len(h_content) > 0:
        left = v_content[0]
        right = v_content[-1]
        top = h_content[0]
        bottom = h_content[-1]
        
        # Crop with small padding
        padding = 10
        left = max(0, left - padding)
        top = max(0, top - padding)
        right = min(im.width, right + padding)
        bottom = min(im.height, bottom + padding)
        
        im = im.crop((left, top, right, bottom))
    
    return im


def robust_autorotate(im: Image.Image) -> Image.Image:
    """Rotate if needed, then trim margins"""
    im = rotate_if_needed(im)
    im = trim_margins_projection(im)
    return im



In [2]:
import sys
print(sys.executable)

/usr/bin/python3


In [ ]:
def pdf_to_images(pdf_path, dpi=DPI):
    """
    Convert PDF pages to images with preprocessing.
    
    Args:
        pdf_path: Path to PDF file
        dpi: DPI for conversion (default: 300)
    
    Returns:
        List of Path objects pointing to temporary image files
    """
    images = []
    pdf_path = Path(pdf_path)

    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)

    with fitz.open(str(pdf_path)) as doc:
        for i, page in enumerate(doc):
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False
            )

            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            img = robust_autorotate(img)
            # Save temp image (needed for vLLM)
            img_path = pdf_path.parent / f"__temp_page_{i+1}.png"
            img.save(img_path)
            images.append(img_path)

    return images


In [ ]:
# !pip install paddlepaddle paddleocr

In [ ]:
from paddleocr import PaddleOCR

paddle_ocr = PaddleOCR(
    use_angle_cls=True,   # handles rotated text
    lang="th",            # change if needed: 'en', 'hi', 'th', etc.
    use_gpu=False         # True if GPU available
)

In [ ]:
def paddle_ocr_text_only(image_path: str) -> str:
    """
    Returns only extracted text from image.
    """
    result = paddle_ocr.ocr(image_path, cls=True)

    lines = []
    for line in result[0]:
        text = line[1][0]
        if text.strip():
            lines.append(text)

    return "\n".join(lines)
    return lines

In [ ]:
import pandas as pd

In [ ]:
def extract_text_from_pdf_paddle_text_only(pdf_path: str) -> str:
    """
    Full OCR pipeline returning plain text only.
    """
    image_paths = pdf_to_images(pdf_path)
    all_text = []

    for img_path in image_paths:
        print(f"OCR page: {img_path}")
        page_text = paddle_ocr_text_only(str(img_path))
        all_text.append(page_text)

    return "\n\n".join(all_text)